# ÉnergiTN - Exploration des données (CRISP-DM : compréhension des données)
Objectif : vérifier que le dataset est réaliste et repérer ce qu'il faudra nettoyer.
Fichiers lus : `../data/synthetic/` (readings, households, weather_used, holidays_used).

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

D = "../data/synthetic/"
r  = pd.read_csv(D + "readings.csv", parse_dates=["timestamp"])
hh = pd.read_csv(D + "households.csv")
w  = pd.read_csv(D + "weather_used.csv", parse_dates=["date"])

r["date"] = r.timestamp.dt.normalize()
df = (r.merge(hh[["household_id", "persona", "region", "ac_coef"]], on="household_id")
        .merge(w[["date", "region", "tmax"]], on=["date", "region"], how="left"))
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": .3})
print(df.shape)
df.head()

(722880, 12)


,household_id,timestamp,kwh,is_holiday,is_outage,is_anomaly,is_synthetic,date,persona,region,ac_coef,tmax
0,H001,2024-01-01 00:00:00,0.113870,1,0,0,True,2024-01-01,etudiant_appartement,Gabes,1.04,18.2
1,H001,2024-01-01 01:00:00,0.096347,1,0,0,True,2024-01-01,etudiant_appartement,Gabes,1.04,18.2
2,H001,2024-01-01 02:00:00,0.088223,1,0,0,True,2024-01-01,etudiant_appartement,Gabes,1.04,18.2
3,H001,2024-01-01 03:00:00,0.100922,1,0,0,True,2024-01-01,etudiant_appartement,Gabes,1.04,18.2
4,H001,2024-01-01 04:00:00,0.106077,1,0,0,True,2024-01-01,etudiant_appartement,Gabes,1.04,18.2


## 1. Contrôle qualité

In [ ]:
print("Foyers            :", df.household_id.nunique())
print("Période           :", df.timestamp.min(), "->", df.timestamp.max())
print("kWh manquants     : {:.2%}".format(df.kwh.isna().mean()))
print("Températures vides: {:.2%}".format(df.tmax.isna().mean()))
print("Anomalies injectées: {:.1%} des heures".format(df.is_anomaly.mean()))
print("Heures de coupure  : {:.2%} (kWh = 0, ce ne sont PAS des valeurs manquantes)".format(df.is_outage.mean()))
print("Doublons (foyer, heure):", df.duplicated(["household_id", "timestamp"]).sum())
print()
print(hh.groupby(["region", "persona"]).size().unstack(fill_value=0))
print()
df.kwh.describe().round(3)

## Graphique 1 : une semaine de consommation d'un foyer (on doit voir les deux pics du jour)

In [ ]:
cand = hh[hh.persona == "famille_5_villa"]
hid = (cand if len(cand) else hh).household_id.iloc[0]
wk = df[(df.household_id == hid) & (df.timestamp >= "2025-07-14") & (df.timestamp < "2025-07-21")]
fig, ax = plt.subplots()
ax.plot(wk.timestamp, wk.kwh, lw=1.2)
ax.set(title=f"Foyer {hid} - semaine du 14 au 20 juillet 2025", ylabel="kWh par heure")
plt.show()

## Graphique 2 : consommation moyenne par mois (le pic d'été)

In [ ]:
m = df.groupby(df.timestamp.dt.month).kwh.mean()
fig, ax = plt.subplots()
ax.bar(m.index, m.values, color="#14305c")
ax.set(title="Consommation moyenne par mois (tous foyers, toutes années)", xlabel="Mois", ylabel="kWh par heure", xticks=range(1, 13))
plt.show()

## Graphique 3 : consommation selon la température (plus il fait chaud, plus ça monte)

In [ ]:
daily = (df.groupby(["household_id", "date"]).agg(kwh_day=("kwh", lambda s: s.sum(min_count=20)), tmax=("tmax", "first"))
           .dropna().reset_index())
s = daily.sample(min(6000, len(daily)), random_state=0)
fig, ax = plt.subplots()
ax.scatter(s.tmax, s.kwh_day, s=5, alpha=.25)
ax.set(title="Consommation journalière selon la température max", xlabel="Tmax (°C)", ylabel="kWh par jour")
plt.show()
print("Corrélation Tmax / kWh du jour :", round(daily.tmax.corr(daily.kwh_day), 2))

## Graphique 4 : journée type en été, par profil de foyer

In [ ]:
summer = df[df.timestamp.dt.month.isin([6, 7, 8]) & (df.is_anomaly == 0)]
prof = summer.groupby([summer.timestamp.dt.hour, "persona"]).kwh.mean().unstack()
ax = prof.plot(figsize=(10, 4), lw=2)
ax.set(title="Journée type en été selon le profil", xlabel="Heure", ylabel="kWh par heure", xticks=range(0, 24, 2))
plt.show()

## Graphique 5 : une anomalie injectée (en rouge)

In [ ]:
a = df[(df.household_id == hid) & (df.is_anomaly == 1)]
if len(a):
    t0 = a.timestamp.min()
    win = df[(df.household_id == hid) & (df.timestamp >= t0 - pd.Timedelta(days=3)) & (df.timestamp <= t0 + pd.Timedelta(days=5))]
    fig, ax = plt.subplots()
    ax.plot(win.timestamp, win.kwh, lw=1, color="#14305c")
    an = win[win.is_anomaly == 1]
    ax.scatter(an.timestamp, an.kwh, color="red", s=14, label="anomalie injectée", zorder=3)
    ax.set(title=f"Foyer {hid} - anomalie à partir du {t0:%d/%m/%Y}", ylabel="kWh par heure"); ax.legend()
    plt.show()
else:
    print("Aucune anomalie pour ce foyer : essayez un autre household_id.")

## Coupures d'électricité (délestages en période de chaleur)

In [ ]:
ev = pd.read_csv(D + "outages_used.csv", parse_dates=["start"])
ev["month"] = ev.start.dt.month
print("Coupures régionales :", len(ev), "| durée moyenne : {:.1f} h | max : {} h".format(ev.hours.mean(), ev.hours.max()))
print("Heures de coupure par foyer et par an : {:.0f}".format(df.is_outage.sum() / df.household_id.nunique() / ((df.timestamp.max() - df.timestamp.min()).days / 365)))
fig, axs = plt.subplots(1, 3, figsize=(14, 3.8))
ev.groupby("month").size().reindex(range(1, 13), fill_value=0).plot.bar(ax=axs[0], color="#c0392b", title="Coupures par mois")
ev.groupby("region").size().sort_values().plot.barh(ax=axs[1], color="#14305c", title="Coupures par région")
ev.hours.plot.hist(ax=axs[2], bins=range(1, 12), rwidth=.85, color="#c28a2c", title="Durée des coupures (heures)")
plt.tight_layout(); plt.show()

### Exemple : une coupure dans la courbe d'un foyer (zone rouge = pas de courant)

In [ ]:
o = df[(df.household_id == hid) & (df.is_outage == 1) & (df.timestamp.dt.month.isin([7, 8]))]
if len(o):
    t0 = o.timestamp.iloc[0].normalize()
    win = df[(df.household_id == hid) & (df.timestamp >= t0 - pd.Timedelta(days=1)) & (df.timestamp < t0 + pd.Timedelta(days=3))]
    fig, ax = plt.subplots()
    ax.plot(win.timestamp, win.kwh, lw=1.2, color="#14305c")
    ax.fill_between(win.timestamp, 0, win.kwh.max(), where=win.is_outage == 1, color="red", alpha=.25, label="coupure")
    ax.set(title=f"Foyer {hid} - coupure(s) en été", ylabel="kWh par heure"); ax.legend()
    plt.show()
else:
    print("Pas de coupure estivale pour ce foyer : changez household_id.")

## Effet des jours fériés et des week-ends

In [ ]:
d = daily.merge(df[["household_id", "date", "is_holiday"]].drop_duplicates(["household_id", "date"]), on=["household_id", "date"])
an = df.groupby(["household_id", "date"]).is_anomaly.max().rename("an").reset_index()
d = d.merge(an, on=["household_id", "date"])
d = d[d.an == 0].copy()          # on exclut les jours d'anomalie injectée, qui faussent la moyenne
d["dow"] = d.date.dt.dayofweek
d["month"] = d.date.dt.month
d["type"] = np.where((d.is_holiday == 1) & (d.dow < 5), "férié (semaine)", np.where(d.dow >= 5, "week-end", "semaine"))
# Comparaison équitable : on divise par la consommation d'un jour de semaine ordinaire du MÊME foyer et du MÊME mois
# (sinon les fériés, plutôt en mars-avril, paraîtraient moins consommateurs à cause de la saison, pas du férié).
ref = d[d.type == "semaine"].groupby(["household_id", "month"]).kwh_day.mean().rename("ref")
d = d.merge(ref, on=["household_id", "month"])
d["ratio"] = d.kwh_day / d.ref
print((d.groupby("type").ratio.mean().round(3) - 1).mul(100).round(1).rename("écart vs jour de semaine normal (%)"))

## À retenir pour le rapport
- Le dataset est **synthétique** (simulateur, graine 42) ; la météo est réelle (Open-Meteo) ou simulée selon le fichier utilisé.
- Pic estival visible, lien température / consommation positif.
- Valeurs manquantes à traiter dans `prepare.py` ; les anomalies injectées ne servent qu'à l'**évaluation**.
- Les **coupures** (`is_outage = 1`, kWh = 0) sont plus fréquentes en canicule : à exclure ou à traiter à part dans l'entraînement de la prévision, sinon le modèle apprend des zéros qui ne sont pas de la vraie demande.
- Prochaine étape : nettoyage, features (lags, moyennes glissantes, `is_holiday`), baseline de prévision.